# 00 环境与种子概览

确认 neo4j-e09 可连、不是 v1 库，并浏览 v2 种子的内容。只读，不写库。

路径与连接参数来自 `e09.config`，与 `make check` 共用。

In [ ]:
import pandas as pd
import yaml
from neo4j import GraphDatabase

from e09.config import NEO4J_AUTH, NEO4J_DB, NEO4J_URI, PAPERS, REPO, SEEDS
from e09.env_check import check_files, check_graph

print(REPO)

## 1. 材料、种子与图谱状态

In [ ]:
errors = check_files() + check_graph()
assert not errors, errors

## 2. v2 种子概览

每个种子文件按 (family, kind) 统计节点与关系；格式见 `data/raw/e09-paper-knowledge/seeds/README.md`。

In [ ]:
seeds = {p.stem: yaml.safe_load(p.read_text()) for p in sorted(SEEDS.glob('*.yml'))}

nodes = pd.DataFrame([
    {'file': f, 'ref': n['ref'], 'family': n['labels'][0], 'kind': n['labels'][1],
     'name': n['properties']['name'], 'aliases': len(n['properties'].get('aliases', []))}
    for f, d in seeds.items() for n in d['nodes']
])
nodes.groupby(['file', 'family', 'kind']).size().unstack('file', fill_value=0)

In [ ]:
rels = pd.DataFrame([{'file': f, **r} for f, d in seeds.items() for r in d['relationships']])
rels.groupby(['file', 'type']).size().unstack('file', fill_value=0)